In [228]:
# DATA MATCHING NOTEBOOK

# I have two datasets, one contains the difficulty/tuning/capo/title/artists and the other includes the chords
# In this file my task is to combine these


In [229]:
import numpy as np
import pandas as pd
import os
import kagglehub
from kagglehub import KaggleDatasetAdapter


# Loading in first dataset: Top 850 Guitar Chards
# * Includes the difficulty that is needed

In [230]:
path = kagglehub.dataset_download("thomaskonstantin/top-850-guitar-tabs")
print("Folder:", path)
print("Exists:", os.path.exists(path))
print("Contents:", os.listdir(path))

csv_name = os.listdir(path)[0]
top850 = pd.read_csv(os.path.join(path, csv_name))

top850.head()

Folder: C:\Users\TiffB\.cache\kagglehub\datasets\thomaskonstantin\top-850-guitar-tabs\versions\1
Exists: True
Contents: ['gutiarDB.csv']


,Artist,Song Name,Song Rating,Song Hits,Page Type,Difficulty,Key,Capo,Tuning
0,Jeff Buckley,Hallelujah,"40,045","31,174,526",Chords,novice,Db,1st fret,E A D G B E
1,Ed Sheeran,Perfect,"31,694","25,794,778",Chords,novice,Ab,1st fret,E A D G B E
2,John Legend,All Of Me,"20,169","25,653,362",Chords,novice,Fm,1st fret,E A D G B E
3,Passenger,Let Her Go,"17,267","24,556,593",Chords,novice,Em,7th fret,E A D G B E
4,Led Zeppelin,Stairway To Heaven,"11,839","20,762,763",Tab,intermediate,Am,No Capo,E A D G B E


# As you can see this includes Difficulty, Key, Capo, and Tuning
# All of which are going to be used to predict the difficulty

# The columns Song Rating and Song Hits are not necessary for my project so I am going to drop those columns

In [231]:
top850 = top850.drop(columns = ["Song Rating", "Song Hits"])
top850

,Artist,Song Name,Page Type,Difficulty,Key,Capo,Tuning
0,Jeff Buckley,Hallelujah,Chords,novice,Db,1st fret,E A D G B E
1,Ed Sheeran,Perfect,Chords,novice,Ab,1st fret,E A D G B E
2,John Legend,All Of Me,Chords,novice,Fm,1st fret,E A D G B E
3,Passenger,Let Her Go,Chords,novice,Em,7th fret,E A D G B E
4,Led Zeppelin,Stairway To Heaven,Tab,intermediate,Am,No Capo,E A D G B E
...,...,...,...,...,...,...,...
845,ZZ Top,Beer Drinkers And Hell Raisers,Chords,intermediate,Db,No Capo,E A D G B E
846,Weezer,Weezer Album,Tab,intermediate,Db,No Capo,E A D G B E
847,Misc Soundtrack,Mission Impossible 2 - Nyah,Tab,intermediate,Db,No Capo,E A D G B
848,Radiohead,Fake Plastic Trees,Chords,intermediate,A,No Capo,E A D G B E


In [232]:
# in the future i'm going to need to match artists/titles to another data set so I am going to make them lower so they match
top850["Artist"] = top850["Artist"].str.lower().str.strip()
top850["Song Name"] = top850["Song Name"].str.lower().str.strip()
top850

,Artist,Song Name,Page Type,Difficulty,Key,Capo,Tuning
0,jeff buckley,hallelujah,Chords,novice,Db,1st fret,E A D G B E
1,ed sheeran,perfect,Chords,novice,Ab,1st fret,E A D G B E
2,john legend,all of me,Chords,novice,Fm,1st fret,E A D G B E
3,passenger,let her go,Chords,novice,Em,7th fret,E A D G B E
4,led zeppelin,stairway to heaven,Tab,intermediate,Am,No Capo,E A D G B E
...,...,...,...,...,...,...,...
845,zz top,beer drinkers and hell raisers,Chords,intermediate,Db,No Capo,E A D G B E
846,weezer,weezer album,Tab,intermediate,Db,No Capo,E A D G B E
847,misc soundtrack,mission impossible 2 - nyah,Tab,intermediate,Db,No Capo,E A D G B
848,radiohead,fake plastic trees,Chords,intermediate,A,No Capo,E A D G B E


In [233]:
# There are too many different page types for the project
# We are only dealing with chords so I am going to focus on that
top850['Page Type'].unique()

<StringArray>
['Chords', 'Tab', 'Ukulele', 'Bass', 'Guitar Pro']
Length: 5, dtype: str

In [234]:
top850 = top850[top850["Page Type"] == "Chords"]
top850["Page Type"].unique()
# now that all the page types are the same we can drop this column

<StringArray>
['Chords']
Length: 1, dtype: str

In [235]:
top850.drop(columns="Page Type")

,Artist,Song Name,Difficulty,Key,Capo,Tuning
0,jeff buckley,hallelujah,novice,Db,1st fret,E A D G B E
1,ed sheeran,perfect,novice,Ab,1st fret,E A D G B E
2,john legend,all of me,novice,Fm,1st fret,E A D G B E
3,passenger,let her go,novice,Em,7th fret,E A D G B E
5,jason mraz,im yours,novice,B,4th fret,E A D G B E
...,...,...,...,...,...,...
841,elton john,your song,intermediate,Db,1st fre,E A D G B E
843,naif,benci untuk mencinta,intermediate,D,No Capo,E A D G B E
844,katy perry,teenage dream,intermediate,Db,3rd fre,E A D G B E
845,zz top,beer drinkers and hell raisers,intermediate,Db,No Capo,E A D G B E


In [236]:
# Difficulty has some values that aren't spelled correctly
# It also has a small number of advanced values

top850["Difficulty"].unique()
# must make intermediat=intermediate and novic=novice

<StringArray>
['novice', 'intermediate', 'intermediat', 'advanced', 'novic']
Length: 5, dtype: str

In [237]:
top850["Difficulty"] = top850["Difficulty"].replace({
    "novic": "beginner",
    "novice": "beginner",
    "intermediat": "intermediate",
})
top850["Difficulty"].unique()

<StringArray>
['beginner', 'intermediate', 'advanced']
Length: 3, dtype: str

In [238]:
# since this is what will be predicted, lets look at the counts of each value
top850['Difficulty'].value_counts(dropna=False)
#Yikes ok it looks like we will just be predicting between two, beginner and intermediate songs
#I will drop the advanced row becuase it is not helping with the prediction at all
# this will be intermediate vs beginner prediction

Difficulty
intermediate    362
beginner        317
advanced          1
Name: count, dtype: int64

In [239]:
#I will drop the advanced row becuase it is not helping with the prediction at all
# this will be intermediate vs beginner prediction

top850 = top850[top850["Difficulty"] != "advanced"]
top850


,Artist,Song Name,Page Type,Difficulty,Key,Capo,Tuning
0,jeff buckley,hallelujah,Chords,beginner,Db,1st fret,E A D G B E
1,ed sheeran,perfect,Chords,beginner,Ab,1st fret,E A D G B E
2,john legend,all of me,Chords,beginner,Fm,1st fret,E A D G B E
3,passenger,let her go,Chords,beginner,Em,7th fret,E A D G B E
5,jason mraz,im yours,Chords,beginner,B,4th fret,E A D G B E
...,...,...,...,...,...,...,...
841,elton john,your song,Chords,intermediate,Db,1st fre,E A D G B E
843,naif,benci untuk mencinta,Chords,intermediate,D,No Capo,E A D G B E
844,katy perry,teenage dream,Chords,intermediate,Db,3rd fre,E A D G B E
845,zz top,beer drinkers and hell raisers,Chords,intermediate,Db,No Capo,E A D G B E


In [240]:
# Nest is the key, everything seems fine with this, I'm just curious how many different options there are
print(top850["Key"].unique())
print(top850["Key"].value_counts(dropna=False))

# This is a decent distribution I believe

<StringArray>
[ ' Db',  ' Ab',  ' Fm',  ' Em',   ' B',   ' C',   ' D',  ' Bm',   ' A',
   ' E', ' Bbm',   ' G', ' F#m',   ' F',   'Db',  ' Am',  ' Cm',  ' F#',
  ' Eb',  ' Dm',  ' C#', ' C#m',  ' Bb',  ' Gm', ' Ebm', ' G#m',  ' Gb',
 ' Abm', ' A#m']
Length: 29, dtype: str
Key
Db      151
 C       87
 G       81
 D       48
 Am      39
 A       38
 Em      36
 F       32
 E       20
 Bm      17
 B       15
 Dm      15
 Eb      12
 F#m     11
 Db      10
 Cm       9
 Bb       9
 Ab       8
 Bbm      7
 C#m      7
 Fm       6
 Gm       5
 Ebm      5
 F#       3
 C#       2
 Gb       2
 Abm      2
 G#m      1
 A#m      1
Name: count, dtype: int64


In [241]:
# I want to convert the Capo strings into 0-12 integers
print(top850["Capo"].value_counts(dropna=False))
top850["Capo"] = top850["Capo"].replace("No Capo", 0)



Capo
No Capo      391
 1st fret     56
 2nd fret     55
 3rd fret     49
 4th fret     32
 5th fret     16
 3rd fre      16
 4th fre      15
 2nd fre      12
 1st fre      12
 6th fret      9
 7th fret      5
 5th fre       4
 7th fre       3
 8th fret      2
 6th fre       2
Name: count, dtype: int64


In [242]:
#match the number with the first character in each string
top850["Capo"] = np.where(top850["Capo"] != 0, top850["Capo"].str[1], top850["Capo"])
top850["Capo"] = top850["Capo"].astype(int)
top850

,Artist,Song Name,Page Type,Difficulty,Key,Capo,Tuning
0,jeff buckley,hallelujah,Chords,beginner,Db,1,E A D G B E
1,ed sheeran,perfect,Chords,beginner,Ab,1,E A D G B E
2,john legend,all of me,Chords,beginner,Fm,1,E A D G B E
3,passenger,let her go,Chords,beginner,Em,7,E A D G B E
5,jason mraz,im yours,Chords,beginner,B,4,E A D G B E
...,...,...,...,...,...,...,...
841,elton john,your song,Chords,intermediate,Db,1,E A D G B E
843,naif,benci untuk mencinta,Chords,intermediate,D,0,E A D G B E
844,katy perry,teenage dream,Chords,intermediate,Db,3,E A D G B E
845,zz top,beer drinkers and hell raisers,Chords,intermediate,Db,0,E A D G B E


In [243]:
top850["Tuning"].value_counts()
# Ok as you can see there are a couple different formattings of tunings
# Why are some tunings on ultimate guitar only 5 strings

Tuning
 E A D G B E          548
 E A D G B             60
E A D G B E            50
 Eb Ab Db Gb Bb Eb     15
 Eb Ab Db Gb Bb E       2
 D G C F A              2
 C# F# B E G# C#        1
 D A D G B              1
Name: count, dtype: int64

In [ ]:
# I actually going to drop this column because ~96% of the tunings are in standard tuning and some of the foramtting is really wonky
# I think it would be too much effort for very little results

top850 = top850.drop(columns="Tuning")
top850

,Artist,Song Name,Page Type,Difficulty,Key,Capo
0,jeff buckley,hallelujah,Chords,beginner,Db,1
1,ed sheeran,perfect,Chords,beginner,Ab,1
2,john legend,all of me,Chords,beginner,Fm,1
3,passenger,let her go,Chords,beginner,Em,7
5,jason mraz,im yours,Chords,beginner,B,4
...,...,...,...,...,...,...
841,elton john,your song,Chords,intermediate,Db,1
843,naif,benci untuk mencinta,Chords,intermediate,D,0
844,katy perry,teenage dream,Chords,intermediate,Db,3
845,zz top,beer drinkers and hell raisers,Chords,intermediate,Db,0


# I have fully cleaned this first dataset
# This dataset will mainly be used for the difficulty of the song, as well as supplemental featurees (key/capo)
# Now I'm moving on to the next data set 
# This set will focus on the chords of the songs

In [ ]:
path = kagglehub.dataset_download("thomaskonstantin/top-850-guitar-tabs")
print("Folder:", path)
print("Exists:", os.path.exists(path))
print("Contents:", os.listdir(path))

csv_name = os.listdir(path)[0]
top850 = pd.read_csv(os.path.join(path, csv_name))

top850.head()